In [18]:
import pandas as pd
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.chunk import tree2conlltags
import uuid
import ast
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, ClassLabel, Features, Sequence, Value
from sklearn.model_selection import train_test_split



In [19]:
# Make sure these are downloaded once in your notebook
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')

[nltk_data] Downloading package punkt to /Users/pals/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/pals/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/pals/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to /Users/pals/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/pals/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /Users/pals/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!


True

In [20]:
ROOT_DIR="/Users/pals/MICS/MIDS_266/project/privacy-ner-att"

In [21]:
DATASET=f'{ROOT_DIR}/datasets/synthetic_privacy_dataset.csv'

In [25]:
DATASET=f'{ROOT_DIR}/datasets/refined_privacy_dataset.csv'

In [26]:
# Load  CSV
df = pd.read_csv(DATASET)

In [28]:
df.columns

Index(['text', 'bio_tags'], dtype='object')

In [38]:
for index, row in df.iterrows():
    break
text = row["text"]
tag_list = ast.literal_eval(row["bio_tags"])
tag_map = {token: tag for tag, token in tag_list}

tokens = word_tokenize(text)
pos = pos_tag(tokens)
chunk = tree2conlltags(ne_chunk(pos))
print(tokens)
print(pos)
print(chunk)

['Daniel', 'Perez', 'visited', 'General', 'Hospital', 'in', 'Seattle', 'on', 'November', '15', ',', '2024', '.', 'The', 'doctor', 'diagnosed', 'Daniel', 'Perez', 'with', 'depression', 'and', 'recommended', 'insulin', 'therapy', '.', 'Later', ',', 'a', 'medical', 'bill', 'was', 'issued', 'to', 'Daniel', 'Perez', ',', 'and', 'payment', 'was', 'processed', 'using', 'the', 'registered', 'Social', 'Security', 'Number', '279-38-9055', '.']
[('Daniel', 'NNP'), ('Perez', 'NNP'), ('visited', 'VBD'), ('General', 'NNP'), ('Hospital', 'NNP'), ('in', 'IN'), ('Seattle', 'NNP'), ('on', 'IN'), ('November', 'NNP'), ('15', 'CD'), (',', ','), ('2024', 'CD'), ('.', '.'), ('The', 'DT'), ('doctor', 'NN'), ('diagnosed', 'VBD'), ('Daniel', 'NNP'), ('Perez', 'NNP'), ('with', 'IN'), ('depression', 'NN'), ('and', 'CC'), ('recommended', 'VBD'), ('insulin', 'NN'), ('therapy', 'NN'), ('.', '.'), ('Later', 'NNP'), (',', ','), ('a', 'DT'), ('medical', 'JJ'), ('bill', 'NN'), ('was', 'VBD'), ('issued', 'VBN'), ('to', '

In [7]:
# Prepare output
rows = []
for _, row in df.iterrows():
    text = row["text"]
    tag_list = ast.literal_eval(row["bio_tags"])
    tag_map = {token: tag for tag, token in tag_list}

    tokens = word_tokenize(text)
    pos = pos_tag(tokens)
    chunk = tree2conlltags(ne_chunk(pos))
    
    # Build ner_tags and pii_tags
    ner_tags = []
    pii_tags = []
    for token in tokens:
        bio = tag_map.get(token, "O")
        ner_tags.append(bio)
        pii_tags.append("PII" if bio != "O" else "O")

    # Build sentence ID
    sentence_id = str(uuid.uuid4())

    # Append rows
    for i, (tok, (word, pos_tags_s), chunk_tags_s) in enumerate(zip(tokens, pos, chunk)):
        conll_rows.append({
            "id": sentence_id,
            "tokens": tok,
            "pos_tags": pos_tags_s,
            "chunk_tags": chunk_tags_s,
            "ner_tags": ner_tags[i],
            "pii_tags": pii_tags[i]
        })

In [8]:
conll_rows[:4]

[{'id': '6b52fd35-9412-4de1-9c18-ea9fe7ef95f6',
  'tokens': 'On',
  'pos_tags': 'IN',
  'chunk_tags': ('On', 'IN', 'O'),
  'ner_tags': 'O',
  'pii_tags': 'O'},
 {'id': '6b52fd35-9412-4de1-9c18-ea9fe7ef95f6',
  'tokens': 'May',
  'pos_tags': 'NNP',
  'chunk_tags': ('May', 'NNP', 'O'),
  'ner_tags': 'O',
  'pii_tags': 'O'},
 {'id': '6b52fd35-9412-4de1-9c18-ea9fe7ef95f6',
  'tokens': '13',
  'pos_tags': 'CD',
  'chunk_tags': ('13', 'CD', 'O'),
  'ner_tags': 'O',
  'pii_tags': 'O'},
 {'id': '6b52fd35-9412-4de1-9c18-ea9fe7ef95f6',
  'tokens': ',',
  'pos_tags': ',',
  'chunk_tags': (',', ',', 'O'),
  'ner_tags': 'O',
  'pii_tags': 'O'}]

In [9]:
# Final DataFrame
df = pd.DataFrame(conll_rows)

In [10]:
grouped = df.groupby("id").agg(list).reset_index()

In [11]:
unique_ner_tags = sorted(set(tag for tags in grouped["ner_tags"] for tag in tags))
unique_pos_tags = sorted(set(tag for tags in grouped["pos_tags"] for tag in tags))
unique_chunk_tags = sorted(set(tag for tags in grouped["chunk_tags"] for tag in tags))
unique_pii_tags = sorted(set(tag for tags in grouped["pii_tags"] for tag in tags))

In [12]:
# Create mapping from tag string → int ID
ner_tag2id = {tag: i for i, tag in enumerate(unique_ner_tags)}
pos_tag2id = {tag: i for i, tag in enumerate(unique_pos_tags)}
chunk_tag2id = {tag: i for i, tag in enumerate(unique_chunk_tags)}
pii_tag2id = {tag: i for i, tag in enumerate(unique_pii_tags)}

# Convert string lists to int IDs
grouped["ner_tags"] = grouped["ner_tags"].apply(lambda tags: [ner_tag2id[t] for t in tags])
grouped["pos_tags"] = grouped["pos_tags"].apply(lambda tags: [pos_tag2id[t] for t in tags])
grouped["chunk_tags"] = grouped["chunk_tags"].apply(lambda tags: [chunk_tag2id[t] for t in tags])
grouped["pii_tags"] = grouped["pii_tags"].apply(lambda tags: [pii_tag2id[t] for t in tags])

In [13]:
#Define Features for HuggingFace Dataset
features = Features({
    "id": Value("string"),
    "tokens": Sequence(Value("string")),
    "pos_tags": Sequence(ClassLabel(names=unique_pos_tags)),
    "chunk_tags": Sequence(ClassLabel(names=unique_chunk_tags)),
    "ner_tags": Sequence(ClassLabel(names=unique_ner_tags)),
    "pii_tags": Sequence(ClassLabel(names=unique_pii_tags)),
})


In [14]:
# Convert to HF Dataset
hf_dataset = Dataset.from_pandas(grouped, features=features)

# Train/Validation/Test split
train_test = hf_dataset.train_test_split(test_size=0.2, seed=42)
val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

In [15]:
# Create DatasetDict
hf_dataset = DatasetDict({
    "train": train_test["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

In [16]:
HFDATASET=f'{ROOT_DIR}/datasets/hf_synthetic_privacy_dataset'

In [17]:
# Save to disk
hf_dataset.save_to_disk(HFDATASET)
print(f"Saved Huggingface Dataset to: {HFDATASET}")

Saving the dataset (1/1 shards): 100%|██████████████████████████████████| 10/10 [00:00<00:00, 915.93 examples/s]

Saved Huggingface Dataset to: /Users/pals/MICS/MIDS_266/project/privacy-ner-att/datasets/hf_synthetic_privacy_dataset
